In [30]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, LabelEncoder
#from shiny import App, render, ui
import pandas as pd
import joblib
import streamlit

In [5]:
df = pd.read_csv('data/bank_customer_dataset.csv')

In [6]:
df.head()

,Customer ID,First Name,Last Name,Date of Birth,Country,Education,Annual Income,Marital Status,Gender,Date of First Transaction,Exited,IsActiveMember,Account Balance,Current Account,Saving Account,Mortgage,Credit Score,Has Credit Card,Number of Complaints,Satisfaction Score
0,1000001,Kimberly,Macias,1947-05-08,UK,Bachelor,178399.86,Single with Partner,Male,2008-10-19,No,No,131719.14,No,No,No,836,No,2,4.01
1,1000002,Brandon,Shaffer,1949-01-01,Canada,Bachelor,134760.96,Married,Male,2021-07-02,No,Yes,241877.38,Yes,Yes,No,493,Yes,4,4.77
2,1000003,Samantha,French,1975-11-13,Germany,GCSE,57277.41,Divorced,Male,2006-04-08,Yes,No,67356.95,No,No,No,418,Yes,2,1.07
3,1000004,Luis,Case,2003-06-14,USA,Bachelor,173062.45,Single,Male,2006-07-06,No,Yes,142780.06,No,No,No,514,No,1,1.71
4,1000005,Anthony,Martinez,1964-12-14,UK,Bachelor,158567.15,Married,Male,2009-05-24,No,No,95635.53,No,No,No,408,No,14,3.64


In [7]:
df.shape

(10000, 20)

In [8]:
df.dtypes

Customer ID                    int64
First Name                    object
Last Name                     object
Date of Birth                 object
Country                       object
Education                     object
Annual Income                float64
Marital Status                object
Gender                        object
Date of First Transaction     object
Exited                        object
IsActiveMember                object
Account Balance              float64
Current Account               object
Saving Account                object
Mortgage                      object
Credit Score                   int64
Has Credit Card               object
Number of Complaints           int64
Satisfaction Score           float64
dtype: object

In [9]:
df.describe()

,Customer ID,Annual Income,Account Balance,Credit Score,Number of Complaints,Satisfaction Score
count,1.000000e+04,10000.000000,10000.000000,10000.000000,10000.00000,10000.000000
mean,1.005000e+06,135166.489324,125554.292749,575.161400,9.99420,3.001735
std,2.886896e+03,46170.465261,71616.098866,158.704778,6.06299,1.162970
min,1.000001e+06,24012.010000,442.750000,300.000000,0.00000,1.000000
25%,1.002501e+06,109501.800000,63714.062500,438.000000,5.00000,1.990000
50%,1.005000e+06,144332.670000,125758.545000,577.000000,10.00000,2.990000
75%,1.007500e+06,171647.212500,188197.100000,712.000000,15.00000,4.020000
max,1.010000e+06,199992.080000,249968.820000,849.000000,20.00000,5.000000


### Data Preparation

1. **Date Preparation**

In [11]:
# Step 1: Convert 'Date of Birth' and 'Date of First Transaction' to datetime format
df['Date of Birth'] = pd.to_datetime(df['Date of Birth'], infer_datetime_format=True, errors='coerce')
df['Date of First Transaction'] = pd.to_datetime(df['Date of First Transaction'], infer_datetime_format=True, errors='coerce')

C:\Users\oladi\AppData\Local\Temp\ipykernel_64548\2567823487.py:2: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df['Date of Birth'] = pd.to_datetime(df['Date of Birth'], infer_datetime_format=True, errors='coerce')
C:\Users\oladi\AppData\Local\Temp\ipykernel_64548\2567823487.py:3: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df['Date of First Transaction'] = pd.to_datetime(df['Date of First Transaction'], infer_datetime_format=True, errors='coerce')


In [12]:
# Step 2: Calculate 'Customer Age'
today = pd.to_datetime('today')
df['Customer Age'] = df['Date of Birth'].apply(lambda dob: today.year - dob.year - ((today.month, today.day) < (dob.month, dob.day)))

# Step 3: Calculate 'Customer Tenor'
df['Customer Tenor'] = df['Date of First Transaction'].apply(lambda txn_date: (today - txn_date).days)

2. **Categorical Label Encoding** 

In [17]:
# List of categorical columns to encode
categorical_cols = ['Country', 'Education', 'Marital Status', 'Gender', 
                    'Exited', 'IsActiveMember', 'Current Account', 
                    'Saving Account', 'Mortgage', 'Has Credit Card']

# Initialize the label encoder
le = LabelEncoder()

# Encode categorical columns
for col in categorical_cols:
    df[col] = le.fit_transform(df[col])

3. **Model Train-Test Split**

In [18]:
# Define the feature matrix 'X' and target variable 'y'
X = df.drop(columns=['Customer ID', 'First Name', 'Last Name', 'Date of Birth', 'Date of First Transaction'])  # Drop 'Customer ID' as it is not a feature
y = df['Exited']  # Assuming 'Exited' is the target variable

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Data preparation summary
print("Data Preparation Summary:")
print(f"Training Set: {X_train.shape}, Testing Set: {X_test.shape}")

Data Preparation Summary:
Training Set: (7000, 17), Testing Set: (3000, 17)


In [19]:
# Initialize the models
models = {
    'Logistic Regression': LogisticRegression(random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'SVM': SVC(random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42)
}

In [20]:
# Train and evaluate models
best_model_name = None
best_accuracy = 0
best_model = None

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Model: {name}, Accuracy: {accuracy:.4f}")
    
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_model_name = name
        best_model = model

Model: Logistic Regression, Accuracy: 0.6827
Model: Decision Tree, Accuracy: 1.0000
Model: Random Forest, Accuracy: 1.0000
Model: SVM, Accuracy: 0.6827
Model: Gradient Boosting, Accuracy: 1.0000


In [21]:
# Best Model
print(f"\nBest Model: {best_model_name} with Accuracy: {best_accuracy:.4f}")

# Generate Classification Report and Confusion Matrix for the Best Model
y_pred_best = best_model.predict(X_test)
print("\nClassification Report:\n", classification_report(y_test, y_pred_best))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_best))


Best Model: Decision Tree with Accuracy: 1.0000

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00      2048
           1       1.00      1.00      1.00       952

    accuracy                           1.00      3000
   macro avg       1.00      1.00      1.00      3000
weighted avg       1.00      1.00      1.00      3000


Confusion Matrix:
 [[2048    0]
 [   0  952]]


### Deployment

In [31]:
# Assuming best_model is the model with the highest accuracy
model_filename = 'best_model.pkl'
joblib.dump(best_model, model_filename)
print(f"Model saved as {model_filename}")

Model saved as best_model.pkl
